In [3]:
import requests
from bs4 import BeautifulSoup
import re
import urllib3
import json
import os

# Vibe Coding by Gemini 都是AI電腦寫的
# 本程式旨在抓取 iTHome 鐵人賽「參賽名單」頁面中，單一指定頁碼的詳細資訊。

# 忽略因 verify=False 產生的 SSL 警告
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- 參數設定：請修改此處的年度和頁碼 (只會抓取此頁) ---
# 設置您想要抓取的 iThome 鐵人賽年份
TARGET_YEAR = 2024 
# 設置您想要抓取的單一頁碼
TARGET_PAGE = 1 
# --------------------------------------------------------

# --- 常數設定 ---
# 動態 URL 模板：用於參賽名單頁面
URL_TEMPLATE = "https://ithelp.ithome.com.tw/{year}ironman/signup/list?page={page}"

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Connection': 'keep-alive'
}

# -----------------------------------------------------
# 函式：抓取總參賽人數 (在單頁模式中，此數值僅供參考)
# -----------------------------------------------------
def get_total_contestant_count(soup: BeautifulSoup) -> int:
    """
    抓取頁面中顯示的總參賽者筆數。
    """
    count_element = soup.find('div', class_='ppl-num') or soup.find('div', class_='contestants-list__header') 
    
    if count_element:
        text = count_element.get_text(strip=True)
        match = re.search(r'(\d+)', text)
        if match:
            return int(match.group(1))
            
    return 0 

# -----------------------------------------------------
# 核心函式：抓取單頁參賽名單詳細資料
# -----------------------------------------------------
def scrape_contestant_list(year: int, page: int):
    """
    從 iThome 鐵人賽的參賽名單頁面抓取所有參賽者的詳細資訊。
    
    Returns:
        tuple: (contestants_data, total_count, success_status)
    """
    target_url = URL_TEMPLATE.format(year=year, page=page)
    
    contestants_data = []
    total_count = 0
    success = False

    print(f"==================================================")
    print(f"【開始抓取單一頁面】 頁碼: {page} / 年份: {year}")
    print(f"目標 URL: {target_url}")
    print(f"==================================================")

    try:
        response = requests.get(target_url, headers=HEADERS, verify=False)
        response.raise_for_status() 
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 抓取總人數供報頭顯示
        total_count = get_total_contestant_count(soup)
        
        list_cards = soup.find_all('div', class_='list-card')
        
        if not list_cards:
            print(f"\n[警告] 頁面 {page} 找到 0 筆資料。")
            return [], total_count, True

        # 遍歷每個卡片，提取所需資訊
        for card in list_cards:
            title_link_element = card.find('a', class_='contestants-list__title')
            person_link_element = card.find('a', class_='contestants-list__person')
            status_label = card.find('label', class_='note')
            img_element = card.find('img', class_='w-100')
            date_element = card.find('div', class_='contestants-list__date')
            
            raw_date_text = date_element.get_text(strip=True) if date_element else "N/A"
            cleaned_date = raw_date_text.replace('報名日期：', '').strip()

            entry = {
                "年度": year,
                "頁碼": page, 
                "報名日期": cleaned_date, 
                "主題": card.find('div', class_='tag').get_text(strip=True) if card.find('div', class_='tag') else "N/A",
                "參賽者名稱": person_link_element.find('div', class_='contestants-list__name').get_text(strip=True) if person_link_element and person_link_element.find('div', class_='contestants-list__name') else "N/A",
                "圖像URL": img_element.get('src') if img_element else "N/A",
                "題目": title_link_element.get_text(strip=True) if title_link_element else "N/A",
                "題目URL": title_link_element.get('href') if title_link_element else "N/A",
                "題目簡介": card.find('p', class_='contestants-list__desc').get_text(strip=True) if card.find('p', class_='contestants-list__desc') else "N/A",
                "進度": status_label.get_text(strip=True) if status_label else "N/A"
            }
            
            contestants_data.append(entry)
            
        success = True
        
        return contestants_data, total_count, success

    except requests.exceptions.RequestException as e:
        print(f"\n[錯誤] 網頁請求失敗: {e}")
    except Exception as e:
        print(f"\n[錯誤] 發生其他錯誤: {e}")

    return [], total_count, success

# -----------------------------------------------------
# 主執行區塊
# -----------------------------------------------------
def main():
    
    # 執行單頁資料抓取
    page_data, total_count, success = scrape_contestant_list(TARGET_YEAR, TARGET_PAGE)
    
    if not success:
        print(f"\n[執行結束] 抓取頁碼 {TARGET_PAGE} 失敗。")
        return

    # 輸出摘要
    print("\n" + "=="*25)
    print(f"【單頁抓取摘要】")
    print(f"iTHome 鐵人賽 {TARGET_YEAR} 年")
    print(f"當前頁碼: {TARGET_PAGE}")
    print(f"本頁抓取筆數: {len(page_data)}")
    print(f"報名總人數 (參考): {total_count}")
    print("=="*25)

    if not page_data:
        return

    # 檔案命名與儲存
    output_filename = f"ironman_{TARGET_YEAR}_page{TARGET_PAGE}.json" 
    
    try:
        # 組合最終 JSON 結構
        final_output = {
            f"it 鐵人賽 {TARGET_YEAR} 年 - 頁碼 {TARGET_PAGE} 名單": page_data
        }
        
        # 生成 JSON 字串並寫入檔案
        json_output = json.dumps(final_output, indent=4, ensure_ascii=False)
        
        with open(output_filename, 'w', encoding='utf-8') as f:
            f.write(json_output)
        
        file_path = os.path.abspath(output_filename)
        print(f"\n✅ JSON 檔案已成功儲存至：\n{file_path}")

    except Exception as e:
        print(f"\n[錯誤] 檔案寫入失敗: {e}")

if __name__ == "__main__":
    main()


【開始抓取單一頁面】 頁碼: 1 / 年份: 2024
目標 URL: https://ithelp.ithome.com.tw/2024ironman/signup/list?page=1

【單頁抓取摘要】
iTHome 鐵人賽 2024 年
當前頁碼: 1
本頁抓取筆數: 10
報名總人數 (參考): 1064

✅ JSON 檔案已成功儲存至：
c:\Users\Hsu\Desktop\mylab_2025\vibe-coding\exchange-rate-webapp\ironman_2024_page1.json
